#  Hugging Face Spaces Website UI

**Project:** Predicting Performance Trajectories of Women Micro-Enterprises: An Early Warning Framework


In [1]:
# 0. Setup

from google.colab import drive
from pathlib import Path

import json
import os
import shutil
import sys
import importlib
import importlib.metadata
import warnings

import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings("ignore")

drive.mount("/content/drive")

project_candidates = [
    Path("/content/drive/MyDrive/24088950_MAI_Research"),
    Path("/content/drive/MyDrive/24088950 MAI Research")
]

PROJECT_DIR = None

for candidate in project_candidates:

    deployment_candidate = (
        candidate
        / "outputs"
        / "deployment"
    )

    if deployment_candidate.exists():

        PROJECT_DIR = candidate
        break

if PROJECT_DIR is None:

    raise FileNotFoundError(
        "The Notebook 06 deployment folder was not found. "
        "Run Notebook 06 first."
    )

OUTPUT_DIR = PROJECT_DIR / "outputs"

DEPLOYMENT_DIR = (
    OUTPUT_DIR
    / "deployment"
)

SPACE_DIR = (
    OUTPUT_DIR
    / "huggingface_space_revised"
)

SPACE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_DIR = (
    SPACE_DIR
    / "model"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project folder:", PROJECT_DIR)
print("Deployment folder:", DEPLOYMENT_DIR)
print("Revised Space folder:", SPACE_DIR)

Mounted at /content/drive
Project folder: /content/drive/MyDrive/24088950_MAI_Research
Deployment folder: /content/drive/MyDrive/24088950_MAI_Research/outputs/deployment
Revised Space folder: /content/drive/MyDrive/24088950_MAI_Research/outputs/huggingface_space_revised


## 1. Validate and Load Notebook 06 Deployment Package

In [2]:
# 1. Validate required deployment files

required_files = [
    "final_early_warning_pipeline.joblib",
    "label_encoder.joblib",
    "risk_thresholds.json",
    "deployment_config.json",
    "input_schema.json",
    "intervention_mapping.json",
    "raw_model_features.json",
    "shap_background.npy"
]

missing_files = [
    filename
    for filename in required_files
    if not (
        DEPLOYMENT_DIR
        / filename
    ).exists()
]

if missing_files:

    raise FileNotFoundError(
        "Deployment package is incomplete:\n"
        + "\n".join(
            missing_files
        )
    )

print("PASS: deployment package found.")

PASS: deployment package found.


In [3]:
# 1.1 Load deployment metadata

with open(
    DEPLOYMENT_DIR / "risk_thresholds.json",
    "r"
) as file:
    risk_thresholds = json.load(file)

with open(
    DEPLOYMENT_DIR / "deployment_config.json",
    "r"
) as file:
    deployment_config = json.load(file)

with open(
    DEPLOYMENT_DIR / "input_schema.json",
    "r"
) as file:
    input_schema = json.load(file)

with open(
    DEPLOYMENT_DIR / "intervention_mapping.json",
    "r"
) as file:
    intervention_mapping = json.load(file)

with open(
    DEPLOYMENT_DIR / "raw_model_features.json",
    "r"
) as file:
    raw_model_features = json.load(file)

print("Selected model:", deployment_config["selected_model"])
print("Raw predictors:", len(raw_model_features))
print("Amber threshold:", risk_thresholds["amber_threshold"])
print("Red threshold:", risk_thresholds["red_threshold"])

Selected model: XGBoost
Raw predictors: 18
Amber threshold: 0.055
Red threshold: 0.285


## 2. Decide Which Inputs Belong in Quick Assessment



In [4]:
# 2. Candidate quick-assessment fields

quick_candidates = [
    "Age",
    "Years in Business",
    "Industry/Business Sector",
    "Household Income",
    "Estimated Remaining Balance",
    "digital_platform_count",
    "attendance_rate",
    "behavior_final_z_in_cohort",
    "behavior_avg_gain_per_session",
    "ssm_registered_binary"
]

QUICK_FEATURES = [
    feature
    for feature in quick_candidates
    if feature in raw_model_features
]

SYSTEM_GENERATED_FEATURES = [
    feature
    for feature in raw_model_features
    if (
        "competency" in feature.lower()
        or feature.startswith("delta_")
    )
]

ADVANCED_FEATURES = [
    feature
    for feature in raw_model_features
    if (
        feature not in QUICK_FEATURES
        and feature not in SYSTEM_GENERATED_FEATURES
    )
]

print("Quick Assessment fields:", len(QUICK_FEATURES))

for feature in QUICK_FEATURES:
    print("-", feature)

print()
print("System-generated / hidden fields:", len(SYSTEM_GENERATED_FEATURES))

for feature in SYSTEM_GENERATED_FEATURES:
    print("-", feature)

print()
print("Optional Advanced fields:", len(ADVANCED_FEATURES))

for feature in ADVANCED_FEATURES:
    print("-", feature)

Quick Assessment fields: 10
- Age
- Years in Business
- Industry/Business Sector
- Household Income
- Estimated Remaining Balance
- digital_platform_count
- attendance_rate
- behavior_final_z_in_cohort
- behavior_avg_gain_per_session
- ssm_registered_binary

System-generated / hidden fields: 1
- overall_competency_change_z

Optional Advanced fields: 7
- State
- Education Level
- Working Status
- IG_followers_pre
- FB_followers_pre
- TT_followers_pre
- behavior_avg_deduct_per_session


## 3. How Competency Change Is Handled



In [5]:
# 3. Check whether competency is present

if SYSTEM_GENERATED_FEATURES:

    print(
        "Competency-related model fields detected:"
    )

    for feature in SYSTEM_GENERATED_FEATURES:
        print("-", feature)

    print()
    print(
        "These fields will NOT be requested in the manual Quick Assessment."
    )

else:

    print(
        "No competency-related final model feature was detected."
    )

Competency-related model fields detected:
- overall_competency_change_z

These fields will NOT be requested in the manual Quick Assessment.


## 4. Copy Deployment Assets

In [6]:
# 4. Copy model assets

asset_files = [
    "final_early_warning_pipeline.joblib",
    "label_encoder.joblib",
    "risk_thresholds.json",
    "deployment_config.json",
    "input_schema.json",
    "intervention_mapping.json",
    "raw_model_features.json",
    "shap_background.npy"
]

optional_assets = [
    "global_low_shap_importance.csv",
    "threshold_performance_audit.csv",
    "final_model_selection_review.csv"
]

for filename in asset_files + optional_assets:

    source = DEPLOYMENT_DIR / filename

    if not source.exists():
        continue

    shutil.copy2(
        source,
        MODEL_DIR / filename
    )

print(
    "Assets copied:",
    len(list(MODEL_DIR.iterdir()))
)

Assets copied: 11


## 5. Create Cohort Upload Template

In [7]:
# 5. Create anonymous cohort template

template_row = {}

for feature in raw_model_features:

    settings = input_schema[feature]

    template_row[feature] = settings.get(
        "default"
    )

cohort_template = pd.DataFrame(
    [template_row]
)

TEMPLATE_PATH = (
    SPACE_DIR
    / "cohort_upload_template.csv"
)

cohort_template.to_csv(
    TEMPLATE_PATH,
    index=False
)

display(
    cohort_template
)

print("Saved:", TEMPLATE_PATH)

,Age,State,Education Level,Working Status,Industry/Business Sector,Years in Business,Household Income,Estimated Remaining Balance,ssm_registered_binary,digital_platform_count,IG_followers_pre,FB_followers_pre,TT_followers_pre,attendance_rate,behavior_avg_gain_per_session,behavior_avg_deduct_per_session,behavior_final_z_in_cohort,overall_competency_change_z
0,26.0,Selangor,SPM,Self-Employed,Food & Beverage,5.0,2200.0,-391.0,1.0,3.0,160.0,365.0,1212.5,92.3,6.3,0.0,0.441049,0.051668


Saved: /content/drive/MyDrive/24088950_MAI_Research/outputs/huggingface_space_revised/cohort_upload_template.csv


# Website Design

## 6. Design Principles



In [8]:
# 6. Human-readable labels used by the generated app

DISPLAY_LABELS = {
    "Age": "Age",
    "State": "State",
    "Education Level": "Education Level",
    "Working Status": "Working Status",
    "Industry/Business Sector": "Industry / Business Sector",
    "Years in Business": "Years in Business",
    "Household Income": "Household Income (RM)",
    "Estimated Remaining Balance": "Estimated Remaining Balance (RM)",
    "Total Commitment": "Total Monthly Commitment (RM)",
    "ssm_registered_binary": "SSM Registered",
    "digital_platform_count": "Digital Platforms Used",
    "IG_followers_pre": "Instagram Followers",
    "FB_followers_pre": "Facebook Followers",
    "TT_followers_pre": "TikTok Followers",
    "IG_has_pre_binary": "Uses Instagram",
    "FB_has_pre_binary": "Uses Facebook",
    "TT_has_pre_binary": "Uses TikTok",
    "attendance_rate": "Attendance Rate",
    "behavior_avg_gain_per_session": "Average Positive Engagement per Session",
    "behavior_avg_deduct_per_session": "Average Behaviour Deduction per Session",
    "behavior_final_z_in_cohort": "Programme Engagement Score",
    "overall_competency_change_z": "Competency Change Score"
}

def display_label(feature):

    return DISPLAY_LABELS.get(
        feature,
        feature.replace("_", " ")
    )

for feature in raw_model_features:

    print(
        feature,
        "→",
        display_label(feature)
    )

Age → Age
State → State
Education Level → Education Level
Working Status → Working Status
Industry/Business Sector → Industry / Business Sector
Years in Business → Years in Business
Household Income → Household Income (RM)
Estimated Remaining Balance → Estimated Remaining Balance (RM)
ssm_registered_binary → SSM Registered
digital_platform_count → Digital Platforms Used
IG_followers_pre → Instagram Followers
FB_followers_pre → Facebook Followers
TT_followers_pre → TikTok Followers
attendance_rate → Attendance Rate
behavior_avg_gain_per_session → Average Positive Engagement per Session
behavior_avg_deduct_per_session → Average Behaviour Deduction per Session
behavior_final_z_in_cohort → Programme Engagement Score
overall_competency_change_z → Competency Change Score


## 7. Generate Revised `app.py`

In [9]:
# 7. Write revised app.py

APP_PATH = (
    SPACE_DIR
    / "app.py"
)

app_code = '\nfrom pathlib import Path\nimport json\nimport tempfile\nimport math\n\nimport numpy as np\nimport pandas as pd\nimport joblib\nimport gradio as gr\n\ntry:\n    import shap\nexcept Exception:\n    shap = None\n\n\nBASE_DIR = Path(__file__).resolve().parent\nMODEL_DIR = BASE_DIR / "model"\n\npipeline = joblib.load(\n    MODEL_DIR / "final_early_warning_pipeline.joblib"\n)\n\nlabel_encoder = joblib.load(\n    MODEL_DIR / "label_encoder.joblib"\n)\n\nwith open(\n    MODEL_DIR / "risk_thresholds.json",\n    "r"\n) as f:\n    risk_thresholds = json.load(f)\n\nwith open(\n    MODEL_DIR / "deployment_config.json",\n    "r"\n) as f:\n    deployment_config = json.load(f)\n\nwith open(\n    MODEL_DIR / "input_schema.json",\n    "r"\n) as f:\n    input_schema = json.load(f)\n\nwith open(\n    MODEL_DIR / "intervention_mapping.json",\n    "r"\n) as f:\n    intervention_mapping = json.load(f)\n\nwith open(\n    MODEL_DIR / "raw_model_features.json",\n    "r"\n) as f:\n    raw_features = json.load(f)\n\n\nAMBER_THRESHOLD = float(\n    risk_thresholds["amber_threshold"]\n)\n\nRED_THRESHOLD = float(\n    risk_thresholds["red_threshold"]\n)\n\nCLASS_NAMES = list(\n    label_encoder.classes_\n)\n\nLOW_CODE = CLASS_NAMES.index(\n    "Low"\n)\n\n\nDISPLAY_LABELS = {\n    "Age": "Age",\n    "State": "State",\n    "Education Level": "Education Level",\n    "Working Status": "Working Status",\n    "Industry/Business Sector": "Industry / Business Sector",\n    "Years in Business": "Years in Business",\n    "Household Income": "Household Income (RM)",\n    "Estimated Remaining Balance": "Estimated Remaining Balance (RM)",\n    "Total Commitment": "Total Monthly Commitment (RM)",\n    "ssm_registered_binary": "SSM Registered",\n    "digital_platform_count": "Digital Platforms Used",\n    "IG_followers_pre": "Instagram Followers",\n    "FB_followers_pre": "Facebook Followers",\n    "TT_followers_pre": "TikTok Followers",\n    "IG_has_pre_binary": "Uses Instagram",\n    "FB_has_pre_binary": "Uses Facebook",\n    "TT_has_pre_binary": "Uses TikTok",\n    "attendance_rate": "Attendance Rate",\n    "behavior_avg_gain_per_session": "Average Positive Engagement per Session",\n    "behavior_avg_deduct_per_session": "Average Behaviour Deduction per Session",\n    "behavior_final_z_in_cohort": "Programme Engagement Score",\n    "overall_competency_change_z": "Competency Change Score",\n}\n\n\ndef display_label(feature):\n\n    return DISPLAY_LABELS.get(\n        feature,\n        feature.replace(\n            "_",\n            " "\n        )\n    )\n\n\nQUICK_CANDIDATES = [\n    "Age",\n    "Years in Business",\n    "Industry/Business Sector",\n    "Household Income",\n    "Estimated Remaining Balance",\n    "digital_platform_count",\n    "attendance_rate",\n    "behavior_final_z_in_cohort",\n    "behavior_avg_gain_per_session",\n    "ssm_registered_binary",\n]\n\nQUICK_FEATURES = [\n    feature\n    for feature in QUICK_CANDIDATES\n    if feature in raw_features\n]\n\nSYSTEM_GENERATED_FEATURES = [\n    feature\n    for feature in raw_features\n    if (\n        "competency" in feature.lower()\n        or feature.startswith(\n            "delta_"\n        )\n    )\n]\n\nADVANCED_FEATURES = [\n    feature\n    for feature in raw_features\n    if (\n        feature not in QUICK_FEATURES\n        and feature not in SYSTEM_GENERATED_FEATURES\n    )\n]\n\n\nCUSTOM_CSS = """\n:root {\n    --pink-50: #fff7fb;\n    --pink-100: #ffeaf4;\n    --pink-200: #ffd3e7;\n    --pink-300: #f8aacd;\n    --pink-500: #df5c98;\n    --pink-600: #c94080;\n    --pink-700: #a72e68;\n    --ink: #3d2c37;\n    --muted: #786470;\n    --card: rgba(255,255,255,.95);\n}\n\n.gradio-container {\n    background:\n        radial-gradient(circle at top left, #ffe7f2 0, transparent 35%),\n        linear-gradient(180deg, #fff9fc 0%, #fff 45%, #fff8fb 100%) !important;\n    color: var(--ink);\n    max-width: 1180px !important;\n}\n\n.hero {\n    background: linear-gradient(135deg, #fff 0%, #ffe8f3 100%);\n    border: 1px solid #f6d2e3;\n    border-radius: 28px;\n    padding: 36px;\n    box-shadow: 0 18px 48px rgba(182, 64, 121, .10);\n    margin-bottom: 18px;\n}\n\n.hero h1 {\n    font-size: 42px;\n    margin: 0 0 12px 0;\n    line-height: 1.05;\n    color: #8c285c;\n}\n\n.hero p {\n    font-size: 17px;\n    color: #6e5262;\n    max-width: 780px;\n}\n\n.mini-badge {\n    display: inline-block;\n    background: #fff0f7;\n    color: #a73570;\n    border: 1px solid #f4c6dc;\n    padding: 7px 12px;\n    border-radius: 999px;\n    font-size: 13px;\n    font-weight: 700;\n    margin-bottom: 14px;\n}\n\n.website-card {\n    background: var(--card);\n    border: 1px solid #f1d9e5;\n    border-radius: 22px;\n    padding: 22px;\n    box-shadow: 0 12px 32px rgba(119, 61, 90, .07);\n}\n\n.section-title {\n    color: #902c61;\n    font-size: 24px;\n    font-weight: 800;\n    margin-bottom: 5px;\n}\n\n.section-copy {\n    color: #786470;\n    margin-bottom: 18px;\n}\n\n.trajectory-grid {\n    display: grid;\n    grid-template-columns: repeat(3, minmax(0,1fr));\n    gap: 14px;\n    margin: 18px 0;\n}\n\n.traj-card {\n    border-radius: 20px;\n    padding: 18px;\n    text-align: center;\n    background: #fff;\n    border: 2px solid #f0dce6;\n    transition: .2s ease;\n}\n\n.traj-card.active {\n    border-color: #d84f91;\n    box-shadow: 0 12px 30px rgba(216,79,145,.18);\n    transform: translateY(-2px);\n}\n\n.traj-card svg {\n    width: 86px;\n    height: 86px;\n    margin: auto;\n    display: block;\n}\n\n.traj-name {\n    font-size: 20px;\n    font-weight: 800;\n    margin-top: 8px;\n}\n\n.traj-prob {\n    font-size: 28px;\n    font-weight: 900;\n    color: #9e3569;\n}\n\n.result-panel {\n    background: linear-gradient(135deg, #fff 0%, #fff0f7 100%);\n    border: 1px solid #f0cddd;\n    border-radius: 24px;\n    padding: 22px;\n    margin-top: 12px;\n}\n\n.risk-pill {\n    display: inline-block;\n    padding: 8px 14px;\n    border-radius: 999px;\n    font-weight: 800;\n    margin: 8px 0;\n}\n\n.risk-green {\n    background: #eef9f0;\n    color: #317444;\n}\n\n.risk-amber {\n    background: #fff7dd;\n    color: #8b6a08;\n}\n\n.risk-red {\n    background: #ffe8ee;\n    color: #a32953;\n}\n\n.completeness {\n    background: #fff7fb;\n    border-left: 4px solid #d65393;\n    padding: 12px 14px;\n    border-radius: 10px;\n    color: #6f5362;\n}\n\nfooter {\n    color: #8b7480;\n}\n"""\n\n\ndef assign_risk_band(\n    low_probability\n):\n\n    if (\n        low_probability\n        >= RED_THRESHOLD\n    ):\n        return (\n            "Red - Administrative Review"\n        )\n\n    if (\n        low_probability\n        >= AMBER_THRESHOLD\n    ):\n        return (\n            "Amber - Monitor"\n        )\n\n    return (\n        "Green - Lower Risk"\n    )\n\n\ndef risk_class(\n    risk_band\n):\n\n    if risk_band.startswith(\n        "Red"\n    ):\n        return "risk-red"\n\n    if risk_band.startswith(\n        "Amber"\n    ):\n        return "risk-amber"\n\n    return "risk-green"\n\n\ndef administrative_action(\n    risk_band\n):\n\n    if risk_band.startswith(\n        "Red"\n    ):\n\n        return (\n            "Priority human review is recommended. "\n            "Review programme engagement, business context and support needs."\n        )\n\n    if risk_band.startswith(\n        "Amber"\n    ):\n\n        return (\n            "Continue monitoring and consider a trainer or mentor check-in."\n        )\n\n    return (\n        "Continue routine programme monitoring and normal support."\n    )\n\n\ndef result_meaning(\n    predicted\n):\n\n    meanings = {\n        "High": (\n            "The model sees a pattern more consistent with sustained business improvement. "\n            "Continue normal monitoring while reinforcing the practices that are working."\n        ),\n        "Moderate": (\n            "The business shows a mixed or less stable pattern. "\n            "It may be progressing, but closer monitoring can help identify whether extra support is needed."\n        ),\n        "Low": (\n            "The model sees a pattern associated with an at-risk trajectory. "\n            "This is an early-warning signal for human review, not a final judgement about the entrepreneur."\n        )\n    }\n\n    return meanings.get(\n        predicted,\n        ""\n    )\n\n\ndef svg_icon(\n    trajectory\n):\n\n    if trajectory == "Low":\n\n        return """\n        <svg viewBox="0 0 100 100" aria-label="Low trajectory illustration">\n          <circle cx="50" cy="50" r="46" fill="#fff0f4"/>\n          <path d="M22 30 L43 47 L58 42 L78 70" fill="none" stroke="#c33b6f" stroke-width="8" stroke-linecap="round" stroke-linejoin="round"/>\n          <path d="M66 69 L79 71 L78 58" fill="none" stroke="#c33b6f" stroke-width="7" stroke-linecap="round" stroke-linejoin="round"/>\n        </svg>\n        """\n\n    if trajectory == "Moderate":\n\n        return """\n        <svg viewBox="0 0 100 100" aria-label="Moderate trajectory illustration">\n          <circle cx="50" cy="50" r="46" fill="#fff7df"/>\n          <path d="M18 55 C30 35, 42 70, 54 50 C64 36, 73 52, 82 46" fill="none" stroke="#c5901e" stroke-width="8" stroke-linecap="round"/>\n        </svg>\n        """\n\n    return """\n    <svg viewBox="0 0 100 100" aria-label="High trajectory illustration">\n      <circle cx="50" cy="50" r="46" fill="#edf8ef"/>\n      <path d="M20 72 L40 54 L53 60 L78 30" fill="none" stroke="#3f8c57" stroke-width="8" stroke-linecap="round" stroke-linejoin="round"/>\n      <path d="M65 31 L79 29 L78 43" fill="none" stroke="#3f8c57" stroke-width="7" stroke-linecap="round" stroke-linejoin="round"/>\n    </svg>\n    """\n\n\ndef trajectory_visual(\n    predicted,\n    probabilities\n):\n\n    ordered = [\n        "Low",\n        "Moderate",\n        "High"\n    ]\n\n    cards = []\n\n    for trajectory in ordered:\n\n        active = (\n            "active"\n            if trajectory == predicted\n            else ""\n        )\n\n        probability = float(\n            probabilities.get(\n                trajectory,\n                0\n            )\n        )\n\n        cards.append(\n            f"""\n            <div class="traj-card {active}">\n                {svg_icon(trajectory)}\n                <div class="traj-name">{trajectory}</div>\n                <div class="traj-prob">{probability:.0%}</div>\n            </div>\n            """\n        )\n\n    return (\n        \'<div class="trajectory-grid">\'\n        + "".join(cards)\n        + "</div>"\n    )\n\n\ndef clean_feature_name(\n    feature_name\n):\n\n    text = str(feature_name)\n\n    text = text.replace(\n        "numeric__",\n        ""\n    )\n\n    text = text.replace(\n        "categorical__",\n        ""\n    )\n\n    text = text.replace(\n        "_infrequent_sklearn",\n        " = Other / Rare"\n    )\n\n    return text\n\n\ndef feature_group(\n    feature_name\n):\n\n    text = clean_feature_name(\n        feature_name\n    ).lower()\n\n    if any(\n        key in text\n        for key in [\n            "attendance",\n            "behavior"\n        ]\n    ):\n        return "Engagement / Behavior"\n\n    if any(\n        key in text\n        for key in [\n            "digital",\n            "followers",\n            "instagram",\n            "facebook",\n            "tiktok",\n            "ig_",\n            "fb_",\n            "tt_"\n        ]\n    ):\n        return "Digital Readiness"\n\n    if any(\n        key in text\n        for key in [\n            "competency",\n            "delta"\n        ]\n    ):\n        return "Competency"\n\n    if "ssm" in text:\n        return "Business Formalisation"\n\n    return "Contextual / Baseline"\n\n\nSHAP_EXPLAINER = None\nSHAP_ERROR = None\n\ntry:\n\n    if shap is not None:\n\n        background = np.load(\n            MODEL_DIR / "shap_background.npy"\n        )\n\n        estimator = (\n            pipeline\n            .named_steps[\n                "model"\n            ]\n        )\n\n        transformed_names = (\n            pipeline\n            .named_steps[\n                "preprocess"\n            ]\n            .get_feature_names_out()\n        )\n\n        try:\n\n            SHAP_EXPLAINER = (\n                shap.Explainer(\n                    estimator,\n                    background,\n                    feature_names=transformed_names\n                )\n            )\n\n        except Exception:\n\n            if (\n                estimator.__class__.__name__\n                == "LogisticRegression"\n            ):\n\n                SHAP_EXPLAINER = (\n                    shap.LinearExplainer(\n                        estimator,\n                        background,\n                        feature_names=transformed_names\n                    )\n                )\n\n            else:\n\n                SHAP_EXPLAINER = (\n                    shap.TreeExplainer(\n                        estimator\n                    )\n                )\n\nexcept Exception as error:\n\n    SHAP_ERROR = str(error)\n\n\ndef extract_low_shap(\n    shap_output\n):\n\n    if isinstance(\n        shap_output,\n        list\n    ):\n        return np.asarray(\n            shap_output[\n                LOW_CODE\n            ]\n        )\n\n    values = np.asarray(\n        shap_output.values\n        if hasattr(\n            shap_output,\n            "values"\n        )\n        else shap_output\n    )\n\n    if values.ndim == 2:\n        return values\n\n    if values.ndim == 3:\n\n        if (\n            values.shape[-1]\n            == len(CLASS_NAMES)\n        ):\n            return values[\n                :,\n                :,\n                LOW_CODE\n            ]\n\n        if (\n            values.shape[0]\n            == len(CLASS_NAMES)\n        ):\n            return values[\n                LOW_CODE,\n                :,\n                :\n            ]\n\n    raise ValueError(\n        f"Unexpected SHAP shape: {values.shape}"\n    )\n\n\ndef local_explanation(\n    input_df,\n    top_n=8\n):\n\n    if SHAP_EXPLAINER is None:\n\n        return pd.DataFrame({\n            "Feature": [\n                "XAI explanation unavailable in this runtime."\n            ],\n            "Area": [\n                "Explanation"\n            ],\n            "Effect Toward Low": [\n                np.nan\n            ]\n        })\n\n    transformed = (\n        pipeline\n        .named_steps[\n            "preprocess"\n        ]\n        .transform(\n            input_df[\n                raw_features\n            ]\n        )\n    )\n\n    transformed = np.asarray(\n        transformed\n    )\n\n    names = (\n        pipeline\n        .named_steps[\n            "preprocess"\n        ]\n        .get_feature_names_out()\n    )\n\n    shap_output = (\n        SHAP_EXPLAINER(\n            transformed\n        )\n    )\n\n    low_values = extract_low_shap(\n        shap_output\n    )[0]\n\n    explanation = pd.DataFrame({\n        "Feature": [\n            clean_feature_name(\n                name\n            )\n            for name in names\n        ],\n        "Area": [\n            feature_group(\n                name\n            )\n            for name in names\n        ],\n        "Effect Toward Low":\n            low_values\n    })\n\n    explanation[\n        "Absolute Effect"\n    ] = (\n        explanation[\n            "Effect Toward Low"\n        ]\n        .abs()\n    )\n\n    explanation = (\n        explanation\n        .sort_values(\n            "Absolute Effect",\n            ascending=False\n        )\n        .head(\n            top_n\n        )\n        .drop(\n            columns=[\n                "Absolute Effect"\n            ]\n        )\n    )\n\n    return explanation\n\n\ndef recommendation_table(\n    explanation\n):\n\n    if (\n        "Effect Toward Low"\n        not in explanation.columns\n    ):\n        return pd.DataFrame()\n\n    positive = (\n        explanation[\n            pd.to_numeric(\n                explanation[\n                    "Effect Toward Low"\n                ],\n                errors="coerce"\n            )\n            > 0\n        ]\n        .copy()\n    )\n\n    rows = []\n    used = set()\n\n    for _, row in (\n        positive\n        .sort_values(\n            "Effect Toward Low",\n            ascending=False\n        )\n        .iterrows()\n    ):\n\n        group = row[\n            "Area"\n        ]\n\n        if (\n            group in used\n            or group\n            not in intervention_mapping\n        ):\n            continue\n\n        mapping = (\n            intervention_mapping[\n                group\n            ]\n        )\n\n        rows.append({\n            "Review Area":\n                group,\n            "What to Review":\n                mapping[\n                    "review"\n                ],\n            "Suggested Support":\n                mapping[\n                    "support"\n                ]\n        })\n\n        used.add(\n            group\n        )\n\n        if len(rows) >= 3:\n            break\n\n    if not rows:\n\n        rows.append({\n            "Review Area":\n                "Human Review",\n            "What to Review":\n                "Review the participant\'s wider programme context.",\n            "Suggested Support":\n                "No automatic intervention recommendation is generated."\n        })\n\n    return pd.DataFrame(\n        rows\n    )\n\n\ndef component_for_feature(\n    feature,\n    advanced=False\n):\n\n    settings = input_schema[\n        feature\n    ]\n\n    label = display_label(\n        feature\n    )\n\n    if advanced:\n        label = (\n            label\n            + " (optional)"\n        )\n\n    default = (\n        None\n        if advanced\n        else settings.get(\n            "default"\n        )\n    )\n\n    if (\n        feature\n        == "ssm_registered_binary"\n    ):\n\n        return gr.Radio(\n            choices=[\n                ("No", 0),\n                ("Yes", 1)\n            ],\n            value=(\n                None\n                if advanced\n                else (\n                    int(\n                        settings.get(\n                            "default",\n                            0\n                        )\n                    )\n                )\n            ),\n            label=label\n        )\n\n    if (\n        settings[\n            "type"\n        ]\n        == "categorical"\n    ):\n\n        return gr.Dropdown(\n            choices=settings.get(\n                "choices",\n                []\n            ),\n            value=default,\n            label=label\n        )\n\n    return gr.Number(\n        value=default,\n        label=label\n    )\n\n\ndef prepare_manual_input(\n    feature_values\n):\n\n    row = {\n        feature: np.nan\n        for feature in raw_features\n    }\n\n    supplied_count = 0\n\n    for feature, value in feature_values.items():\n\n        if (\n            value is None\n            or value == ""\n        ):\n            continue\n\n        settings = input_schema[\n            feature\n        ]\n\n        if (\n            settings[\n                "type"\n            ]\n            == "number"\n        ):\n\n            row[\n                feature\n            ] = float(value)\n\n        else:\n\n            row[\n                feature\n            ] = value\n\n        supplied_count += 1\n\n    completeness = (\n        supplied_count\n        / len(\n            raw_features\n        )\n        if raw_features\n        else 0\n    )\n\n    return (\n        pd.DataFrame(\n            [row],\n            columns=raw_features\n        ),\n        completeness,\n        supplied_count\n    )\n\n\ndef predict_dataframe(\n    input_df\n):\n\n    input_df = input_df[\n        raw_features\n    ].copy()\n\n    probabilities = (\n        pipeline\n        .predict_proba(\n            input_df\n        )\n    )\n\n    codes = (\n        pipeline\n        .predict(\n            input_df\n        )\n    )\n\n    labels = (\n        label_encoder\n        .inverse_transform(\n            codes.astype(int)\n        )\n    )\n\n    rows = []\n\n    for index in range(\n        len(input_df)\n    ):\n\n        class_probs = {\n            class_name:\n                float(\n                    probabilities[\n                        index,\n                        class_index\n                    ]\n                )\n            for class_index, class_name\n            in enumerate(\n                CLASS_NAMES\n            )\n        }\n\n        low_probability = (\n            class_probs[\n                "Low"\n            ]\n        )\n\n        risk_band = (\n            assign_risk_band(\n                low_probability\n            )\n        )\n\n        rows.append({\n            "Predicted Trajectory":\n                labels[index],\n            "Probability High":\n                class_probs.get(\n                    "High",\n                    np.nan\n                ),\n            "Probability Moderate":\n                class_probs.get(\n                    "Moderate",\n                    np.nan\n                ),\n            "Probability Low":\n                low_probability,\n            "Risk Band":\n                risk_band\n        })\n\n    return pd.DataFrame(\n        rows\n    )\n\n\ndef individual_analysis(\n    *values\n):\n\n    all_manual_features = (\n        QUICK_FEATURES\n        + ADVANCED_FEATURES\n    )\n\n    feature_values = {\n        feature: value\n        for feature, value\n        in zip(\n            all_manual_features,\n            values\n        )\n    }\n\n    input_df, completeness, supplied_count = (\n        prepare_manual_input(\n            feature_values\n        )\n    )\n\n    prediction = (\n        predict_dataframe(\n            input_df\n        )\n        .iloc[0]\n    )\n\n    predicted = (\n        prediction[\n            "Predicted Trajectory"\n        ]\n    )\n\n    probabilities = {\n        "High":\n            float(\n                prediction[\n                    "Probability High"\n                ]\n            ),\n        "Moderate":\n            float(\n                prediction[\n                    "Probability Moderate"\n                ]\n            ),\n        "Low":\n            float(\n                prediction[\n                    "Probability Low"\n                ]\n            )\n    }\n\n    risk_band = (\n        prediction[\n            "Risk Band"\n        ]\n    )\n\n    visual = (\n        trajectory_visual(\n            predicted,\n            probabilities\n        )\n    )\n\n    risk_css = risk_class(\n        risk_band\n    )\n\n    completeness_message = (\n        f"""\n        <div class="completeness">\n        <b>Data completeness:</b> {supplied_count} of {len(raw_features)} model fields provided ({completeness:.0%}).\n        """\n    )\n\n    if completeness < 1:\n\n        completeness_message += (\n            "<br>This is a <b>preliminary screening</b>. "\n            "Missing model fields are handled by the fitted preprocessing pipeline. "\n            "For a fuller assessment, use Advanced Information or cohort data."\n        )\n\n    completeness_message += (\n        "</div>"\n    )\n\n    summary = f"""\n    <div class="result-panel">\n        <div class="mini-badge">Early-Warning Result</div>\n        <h2 style="margin:4px 0;color:#8d2d60;">{predicted} Trajectory</h2>\n        <span class="risk-pill {risk_css}">{risk_band}</span>\n        <p><b>What this means</b><br>{result_meaning(predicted)}</p>\n        <p><b>Recommended next step</b><br>{administrative_action(risk_band)}</p>\n        {completeness_message}\n        <p style="font-size:13px;color:#806c77;margin-top:14px;">\n        This model supports human review. It must not automatically determine programme eligibility, funding, graduation or exclusion.\n        </p>\n    </div>\n    """\n\n    explanation = (\n        local_explanation(\n            input_df,\n            top_n=10\n        )\n    )\n\n    recommendations = (\n        recommendation_table(\n            explanation\n        )\n    )\n\n    explanation_help = """\n    ### Why did the model produce this result?\n\n    The XAI table shows which model features had the strongest influence on this participant\'s **Low-trajectory probability**.\n\n    - Positive values push the prediction more toward **Low**.\n    - Negative values push the prediction away from **Low**.\n    - These are model contributions, not proof that a factor causes poor business performance.\n    """\n\n    return (\n        visual,\n        summary,\n        probabilities,\n        explanation_help,\n        explanation,\n        recommendations\n    )\n\n\nPII_KEYWORDS = [\n    "name",\n    "full name",\n    "ic",\n    "nric",\n    "phone",\n    "contact",\n    "address",\n    "ssm number",\n    "social handle",\n    "instagram handle",\n    "facebook handle",\n    "tiktok handle"\n]\n\n\ndef cohort_analysis(\n    uploaded_file\n):\n\n    if uploaded_file is None:\n\n        raise gr.Error(\n            "Please upload a CSV file."\n        )\n\n    df = pd.read_csv(\n        uploaded_file\n    )\n\n    detected_pii = []\n\n    for column in df.columns:\n\n        lower = str(\n            column\n        ).strip().lower()\n\n        if any(\n            keyword in lower\n            for keyword in PII_KEYWORDS\n        ):\n\n            if column not in raw_features:\n\n                detected_pii.append(\n                    column\n                )\n\n    if detected_pii:\n\n        raise gr.Error(\n            "Potential identifying columns detected: "\n            + ", ".join(\n                detected_pii\n            )\n            + ". Remove them before analysis."\n        )\n\n    missing = [\n        feature\n        for feature in raw_features\n        if feature not in df.columns\n    ]\n\n    if missing:\n\n        raise gr.Error(\n            "Missing required model columns: "\n            + ", ".join(\n                missing\n            )\n        )\n\n    scored = (\n        predict_dataframe(\n            df[\n                raw_features\n            ]\n        )\n    )\n\n    output = pd.concat(\n        [\n            df.reset_index(\n                drop=True\n            ),\n            scored\n        ],\n        axis=1\n    )\n\n    counts = (\n        output[\n            "Risk Band"\n        ]\n        .value_counts()\n    )\n\n    green = int(\n        counts.get(\n            "Green - Lower Risk",\n            0\n        )\n    )\n\n    amber = int(\n        counts.get(\n            "Amber - Monitor",\n            0\n        )\n    )\n\n    red = int(\n        counts.get(\n            "Red - Administrative Review",\n            0\n        )\n    )\n\n    summary = f"""\n    <div class="website-card">\n        <div class="section-title">Cohort Summary</div>\n        <p><b>Participants analysed:</b> {len(output)}</p>\n        <p>🟢 <b>Green — Lower Risk:</b> {green}</p>\n        <p>🟡 <b>Amber — Monitor:</b> {amber}</p>\n        <p>🔴 <b>Red — Administrative Review:</b> {red}</p>\n        <p><b>Average Low probability:</b> {output[\'Probability Low\'].mean():.1%}</p>\n    </div>\n    """\n\n    temp_file = (\n        tempfile.NamedTemporaryFile(\n            suffix=".csv",\n            delete=False\n        )\n    )\n\n    temp_file.close()\n\n    output.to_csv(\n        temp_file.name,\n        index=False\n    )\n\n    preview = output[\n        [\n            "Predicted Trajectory",\n            "Probability High",\n            "Probability Moderate",\n            "Probability Low",\n            "Risk Band"\n        ]\n    ].copy()\n\n    return (\n        summary,\n        preview,\n        temp_file.name\n    )\n\n\nglobal_importance_path = (\n    MODEL_DIR\n    / "global_low_shap_importance.csv"\n)\n\nif global_importance_path.exists():\n\n    global_importance = pd.read_csv(\n        global_importance_path\n    ).head(15)\n\nelse:\n\n    global_importance = pd.DataFrame({\n        "Feature": [\n            "Run Notebook 06 to export global SHAP importance."\n        ],\n        "Mean Absolute SHAP": [\n            np.nan\n        ]\n    })\n\n\npink_theme = gr.themes.Soft(\n    primary_hue="pink",\n    secondary_hue="rose",\n    neutral_hue="slate"\n)\n\n\nwith gr.Blocks(\n    theme=pink_theme,\n    css=CUSTOM_CSS,\n    title="Ibupreneur Early-Warning System"\n) as demo:\n\n    with gr.Tabs():\n\n        with gr.Tab(\n            "Home"\n        ):\n\n            gr.HTML(\n                """\n                <div class="hero">\n                    <div class="mini-badge">Ibupreneur Academy • Decision Support</div>\n                    <h1>Early signals.<br>Earlier support.</h1>\n                    <p>\n                    An interpretable early-warning prototype that helps programme administrators identify\n                    women micro-entrepreneurs who may benefit from closer monitoring or additional support.\n                    </p>\n                </div>\n                """\n            )\n\n            with gr.Row():\n\n                gr.HTML(\n                    """\n                    <div class="website-card">\n                        <div class="section-title">01. Assess</div>\n                        <p class="section-copy">\n                        Complete a short screening using information administrators already know.\n                        </p>\n                    </div>\n                    """\n                )\n\n                gr.HTML(\n                    """\n                    <div class="website-card">\n                        <div class="section-title">02. Understand</div>\n                        <p class="section-copy">\n                        See High, Moderate and Low probabilities together with an XAI explanation.\n                        </p>\n                    </div>\n                    """\n                )\n\n                gr.HTML(\n                    """\n                    <div class="website-card">\n                        <div class="section-title">03. Support</div>\n                        <p class="section-copy">\n                        Use the result to guide human review and targeted programme support.\n                        </p>\n                    </div>\n                    """\n                )\n\n            gr.Markdown(\n                """\n### How to use this prototype\n\nGo to **Individual Assessment** for one participant, or **Cohort Monitoring** for a prepared cohort CSV.\n\nThe model does **not** make final decisions. It provides an early-warning signal for programme staff.\n"""\n            )\n\n        with gr.Tab(\n            "Individual Assessment"\n        ):\n\n            gr.HTML(\n                """\n                <div class="website-card">\n                    <div class="section-title">Quick Assessment</div>\n                    <div class="section-copy">\n                    Start with a small number of easy-to-understand fields. Advanced information is optional.\n                    The competency change z-score is not requested manually because it is a system-generated research feature.\n                    </div>\n                </div>\n                """\n            )\n\n            quick_components = []\n\n            with gr.Row():\n\n                with gr.Column():\n\n                    for feature in (\n                        QUICK_FEATURES[\n                            : math.ceil(\n                                len(\n                                    QUICK_FEATURES\n                                ) / 2\n                            )\n                        ]\n                    ):\n\n                        component = (\n                            component_for_feature(\n                                feature,\n                                advanced=False\n                            )\n                        )\n\n                        quick_components.append(\n                            component\n                        )\n\n                with gr.Column():\n\n                    for feature in (\n                        QUICK_FEATURES[\n                            math.ceil(\n                                len(\n                                    QUICK_FEATURES\n                                ) / 2\n                            ):\n                        ]\n                    ):\n\n                        component = (\n                            component_for_feature(\n                                feature,\n                                advanced=False\n                            )\n                        )\n\n                        quick_components.append(\n                            component\n                        )\n\n            advanced_components = []\n\n            if ADVANCED_FEATURES:\n\n                with gr.Accordion(\n                    "Optional Advanced Information",\n                    open=False\n                ):\n\n                    gr.Markdown(\n                        """\nAdd these only if the information is readily available. Leaving them blank is allowed for preliminary screening.\n"""\n                    )\n\n                    for feature in ADVANCED_FEATURES:\n\n                        component = (\n                            component_for_feature(\n                                feature,\n                                advanced=True\n                            )\n                        )\n\n                        advanced_components.append(\n                            component\n                        )\n\n            if SYSTEM_GENERATED_FEATURES:\n\n                gr.Markdown(\n                    """\n> **Competency change:** not requested manually. In the research workflow it is derived from pre/post assessment data. During quick manual screening, if it is unavailable, the fitted preprocessing pipeline handles it as missing. A fuller production system should calculate it automatically from Academy assessment records.\n"""\n                )\n\n            analyse_button = gr.Button(\n                "Analyse Early-Warning Risk",\n                variant="primary"\n            )\n\n            gr.HTML(\n                \'<div class="section-title" style="margin-top:28px;">Your Result</div>\'\n            )\n\n            trajectory_output = gr.HTML()\n\n            result_summary = gr.HTML()\n\n            probability_output = gr.Label(\n                label="Trajectory Probabilities",\n                num_top_classes=3\n            )\n\n            xai_help = gr.Markdown()\n\n            explanation_output = gr.Dataframe(\n                headers=[\n                    "Feature",\n                    "Area",\n                    "Direction"\n   "Strength"\n   "Effect Toward Low"\n                  ],\n                label="XAI: Main Model Contributions",\n                interactive=False\n            )\n\n            recommendation_output = gr.Dataframe(\n                headers=[\n                    "Review Area",\n                    "What to Review",\n                    "Suggested Support"\n                ],\n                label="Suggested Administrative Review",\n                interactive=False\n            )\n\n            analyse_button.click(\n                fn=individual_analysis,\n                inputs=(\n                    quick_components\n                    + advanced_components\n                ),\n                outputs=[\n                    trajectory_output,\n                    result_summary,\n                    probability_output,\n                    xai_help,\n                    explanation_output,\n                    recommendation_output\n                ]\n            )\n\n        with gr.Tab(\n            "Cohort Monitoring"\n        ):\n\n            gr.HTML(\n                """\n                <div class="website-card">\n                    <div class="section-title">Cohort Monitoring</div>\n                    <div class="section-copy">\n                    Upload a prepared research/admin CSV to score multiple participants and identify cases for review.\n                    </div>\n                </div>\n                """\n            )\n\n            gr.Markdown(\n                """\nUse `cohort_upload_template.csv` as the structure.\n\nDo not upload names, IC / NRIC numbers, phone numbers, addresses, SSM numbers or social-media handles.\n"""\n            )\n\n            cohort_file = gr.File(\n                label="Upload cohort CSV",\n                file_types=[".csv"],\n                type="filepath"\n            )\n\n            cohort_button = gr.Button(\n                "Analyse Cohort",\n                variant="primary"\n            )\n\n            cohort_summary = gr.HTML()\n\n            cohort_preview = gr.Dataframe(\n                label="Scored Cohort Preview",\n                interactive=False\n            )\n\n            cohort_download = gr.File(\n                label="Download Scored CSV"\n            )\n\n            cohort_button.click(\n                fn=cohort_analysis,\n                inputs=[cohort_file],\n                outputs=[\n                    cohort_summary,\n                    cohort_preview,\n                    cohort_download\n                ]\n            )\n\n        with gr.Tab(\n            "XAI & How It Works"\n        ):\n\n            gr.HTML(\n                """\n                <div class="hero">\n                    <div class="mini-badge">Explainable AI</div>\n                    <h1 style="font-size:34px;">Understand the signal,<br>not just the label.</h1>\n                    <p>\n                    The prototype uses model explanation to show which variables most influenced the Low-trajectory probability.\n                    </p>\n                </div>\n                """\n            )\n\n            gr.Markdown(\n                """\n### How to read an individual XAI result\n\nA **positive** contribution pushes the fitted model more toward the Low trajectory.\n\nA **negative** contribution pushes it away from Low.\n\nThe explanation describes the model\'s mathematical prediction. It does **not** establish causation.\n\n### Competency change\n\n`overall_competency_change_z` is an engineered research variable. Programme users should not have to calculate or type it manually.\n\nIn a future integrated system:\n\n**pre/post assessment records → automatic competency calculation → model**\n\n### Early-warning bands\n\n**Green — Lower Risk**  \nRoutine monitoring.\n\n**Amber — Monitor**  \nCloser monitoring or trainer check-in may be useful.\n\n**Red — Administrative Review**  \nPriority human review is recommended.\n\nNo band automatically determines eligibility, funding, graduation or exclusion.\n"""\n            )\n\n            gr.Dataframe(\n                value=global_importance,\n                label="Global Features Influencing Low-Trajectory Predictions",\n                interactive=False\n            )\n\n    gr.Markdown(\n        f"""\n---\n\n**Model:** {deployment_config[\'selected_model\']}  \n**Amber threshold:** {AMBER_THRESHOLD:.3f}  \n**Red threshold:** {RED_THRESHOLD:.3f}\n\nResearch prototype for administrative decision support with human oversight.\n"""\n    )\n\n\nif __name__ == "__main__":\n    demo.launch()\n'

APP_PATH.write_text(
    app_code,
    encoding="utf-8"
)

print("Created:", APP_PATH)
print("Lines:", len(app_code.splitlines()))

Created: /content/drive/MyDrive/24088950_MAI_Research/outputs/huggingface_space_revised/app.py
Lines: 1767


In [10]:
# 7.1 Fix XGBoost SHAP handling

app_text = APP_PATH.read_text(
    encoding="utf-8"
)

# Replace SHAP explainer initialization
start = app_text.index(
    "SHAP_EXPLAINER = None"
)

end = app_text.index(
    "\ndef extract_low_shap",
    start
)

new_shap_setup = r'''
SHAP_EXPLAINER = None
SHAP_ERROR = None
SHAP_MODE = None

try:

    if shap is not None:

        background = np.load(
            MODEL_DIR / "shap_background.npy"
        )

        estimator = (
            pipeline
            .named_steps[
                "model"
            ]
        )

        transformed_names = (
            pipeline
            .named_steps[
                "preprocess"
            ]
            .get_feature_names_out()
        )

        if (
            estimator.__class__.__name__
            == "XGBClassifier"
        ):

            SHAP_EXPLAINER = (
                shap.TreeExplainer(
                    estimator,
                    feature_perturbation=(
                        "tree_path_dependent"
                    )
                )
            )

            SHAP_MODE = (
                "XGBoost TreeSHAP"
            )

        elif (
            estimator.__class__.__name__
            == "LogisticRegression"
        ):

            SHAP_EXPLAINER = (
                shap.LinearExplainer(
                    estimator,
                    background,
                    feature_names=(
                        transformed_names
                    )
                )
            )

            SHAP_MODE = (
                "Linear SHAP"
            )

        else:

            SHAP_EXPLAINER = (
                shap.TreeExplainer(
                    estimator
                )
            )

            SHAP_MODE = (
                "TreeSHAP"
            )

except Exception as error:

    SHAP_ERROR = str(
        error
    )

    print(
        "Primary SHAP setup error:",
        repr(error)
    )
'''

app_text = (
    app_text[:start]
    + new_shap_setup.strip()
    + "\n\n"
    + app_text[end + 1:]
)


# Replace local_explanation function
start = app_text.index(
    "def local_explanation("
)

end = app_text.index(
    "\ndef recommendation_table",
    start
)

new_local_explanation = r'''
def local_explanation(
    input_df,
    top_n=8
):

    transformed = (
        pipeline
        .named_steps[
            "preprocess"
        ]
        .transform(
            input_df[
                raw_features
            ]
        )
    )

    transformed = np.asarray(
        transformed
    )

    names = (
        pipeline
        .named_steps[
            "preprocess"
        ]
        .get_feature_names_out()
    )

    shap_output = None

    # First try the model-specific SHAP explainer
    try:

        if SHAP_EXPLAINER is not None:

            shap_output = (
                SHAP_EXPLAINER(
                    transformed
                )
            )

    except Exception as error:

        print(
            "TreeSHAP error:",
            repr(error)
        )

        shap_output = None


    # Fallback for XGBoost / categorical split issues
    if shap_output is None:

        try:

            fallback_background = np.load(
                MODEL_DIR
                / "shap_background.npy"
            )

            estimator = (
                pipeline
                .named_steps[
                    "model"
                ]
            )

            fallback_explainer = (
                shap.PermutationExplainer(
                    estimator.predict_proba,
                    fallback_background,
                    feature_names=names
                )
            )

            minimum_evals = (
                2
                * transformed.shape[1]
                + 1
            )

            shap_output = (
                fallback_explainer(
                    transformed,
                    max_evals=max(
                        minimum_evals,
                        101
                    )
                )
            )

            print(
                "Using model-agnostic "
                "Permutation SHAP fallback."
            )

        except Exception as error:

            print(
                "Fallback SHAP error:",
                repr(error)
            )

            return pd.DataFrame({
                "Feature": [
                    (
                        "Prediction successful, "
                        "but XAI explanation "
                        "could not be generated."
                    )
                ],
                "Area": [
                    "Explanation"
                ],
                "Effect Toward Low": [
                    np.nan
                ]
            })


    try:

        low_values = (
            extract_low_shap(
                shap_output
            )[0]
        )

        explanation = pd.DataFrame({
            "Feature": [
                clean_feature_name(
                    name
                )
                for name in names
            ],
            "Area": [
                feature_group(
                    name
                )
                for name in names
            ],
            "Effect Toward Low":
                low_values
        })

        explanation[
            "Absolute Effect"
        ] = (
            explanation[
                "Effect Toward Low"
            ]
            .abs()
        )

        explanation = (
            explanation
            .sort_values(
                "Absolute Effect",
                ascending=False
            )
            .head(
                top_n
            )
            .drop(
                columns=[
                    "Absolute Effect"
                ]
            )
        )

        return explanation

    except Exception as error:

        print(
            "XAI formatting error:",
            repr(error)
        )

        return pd.DataFrame({
            "Feature": [
                (
                    "Prediction successful, "
                    "but the XAI result "
                    "could not be formatted."
                )
            ],
            "Area": [
                "Explanation"
            ],
            "Effect Toward Low": [
                np.nan
            ]
        })
'''

app_text = (
    app_text[:start]
    + new_local_explanation.strip()
    + "\n\n"
    + app_text[end + 1:]
)

APP_PATH.write_text(
    app_text,
    encoding="utf-8"
)

print(
    "PASS: XGBoost SHAP patch applied."
)

PASS: XGBoost SHAP patch applied.


In [11]:
# 7.2 Validate patched app.py

source_code = APP_PATH.read_text(
    encoding="utf-8"
)

compile(
    source_code,
    str(APP_PATH),
    "exec"
)

print(
    "PASS: patched app.py syntax is valid."
)

PASS: patched app.py syntax is valid.


In [12]:
# 7.3 Add descriptions to every input field

app_text = APP_PATH.read_text(
    encoding="utf-8"
)

feature_info_code = r'''

FEATURE_INFO = {

    "Age":
        "Participant's age in completed years. Example: 38.",

    "State":
        "State where the participant or business is based.",

    "Education Level":
        "Highest education level recorded for the participant.",

    "Working Status":
        "Participant's current employment or working status.",

    "Industry/Business Sector":
        "Main sector of the participant's business.",

    "Years in Business":
        "Number of years the participant has operated the business. "
        "Example: enter 3 for three years.",

    "Household Income":
        "Household income recorded in the Academy dataset, in RM. "
        "Use the same reporting period used in the participant record.",

    "Estimated Remaining Balance":
        "Estimated RM amount remaining after household commitments or expenses, "
        "based on the participant's recorded financial information. "
        "A negative value means commitments exceed the available amount.",

    "ssm_registered_binary":
        "Whether the participant's business is registered with SSM. "
        "No = 0, Yes = 1.",

    "digital_platform_count":
        "Number of business digital platforms used at baseline. "
        "This model counts Instagram, Facebook and TikTok, so the value ranges from 0 to 3.",

    "IG_followers_pre":
        "Number of Instagram followers recorded at baseline, before the outcome monitoring period.",

    "FB_followers_pre":
        "Number of Facebook followers recorded at baseline, before the outcome monitoring period.",

    "TT_followers_pre":
        "Number of TikTok followers recorded at baseline, before the outcome monitoring period.",

    "attendance_rate":
        "Programme attendance percentage from 0 to 100. "
        "Example: enter 85 for 85% attendance.",

    "behavior_avg_gain_per_session":
        "Average positive Behaviour Points awarded per attended session. "
        "This comes from the trainer's Behaviour Points records.",

    "behavior_avg_deduct_per_session":
        "Average Behaviour Points deducted per session. "
        "Higher values mean more deductions were recorded.",

    "behavior_final_z_in_cohort":
        "System-generated engagement score comparing the participant's final Behaviour Points "
        "with others in the same cohort. "
        "0 = cohort average, +1 = about one standard deviation above average, "
        "-1 = about one standard deviation below average.",

    "overall_competency_change_z":
        "System-generated competency improvement score derived from pre/post assessments. "
        "0 = around the cohort average change, positive = greater improvement, "
        "negative = lower improvement relative to the cohort. "
        "Administrators should not calculate this manually."
}


def feature_description(feature):

    return FEATURE_INFO.get(
        feature,
        "Use the value recorded in the participant's programme dataset."
    )
'''

marker = "\ndef display_label(feature):"

if "FEATURE_INFO = {" not in app_text:

    position = app_text.index(
        marker
    )

    app_text = (
        app_text[:position]
        + "\n"
        + feature_info_code
        + "\n"
        + app_text[position:]
    )


start = app_text.index(
    "def component_for_feature("
)

end = app_text.index(
    "\ndef prepare_manual_input",
    start
)

new_component_function = r'''
def component_for_feature(
    feature,
    advanced=False
):

    settings = input_schema[
        feature
    ]

    label = display_label(
        feature
    )

    description = (
        feature_description(
            feature
        )
    )

    if advanced:

        label = (
            label
            + " (optional)"
        )

    default = (
        None
        if advanced
        else settings.get(
            "default"
        )
    )

    if (
        feature
        == "ssm_registered_binary"
    ):

        return gr.Radio(
            choices=[
                ("No", 0),
                ("Yes", 1)
            ],
            value=(
                None
                if advanced
                else int(
                    settings.get(
                        "default",
                        0
                    )
                )
            ),
            label=label,
            info=description
        )

    if (
        settings[
            "type"
        ]
        == "categorical"
    ):

        return gr.Dropdown(
            choices=settings.get(
                "choices",
                []
            ),
            value=default,
            label=label,
            info=description
        )

    return gr.Number(
        value=default,
        label=label,
        info=description
    )
'''

app_text = (
    app_text[:start]
    + new_component_function.strip()
    + "\n\n"
    + app_text[end + 1:]
)

APP_PATH.write_text(
    app_text,
    encoding="utf-8"
)

print(
    "PASS: input descriptions added."
)

PASS: input descriptions added.


In [13]:
# 7.4 Improve XAI table

import re

app_text = APP_PATH.read_text(
    encoding="utf-8"
)

start = app_text.index(
    "def local_explanation("
)

end = app_text.index(
    "\ndef recommendation_table",
    start
)

local_function = app_text[
    start:end
]

old_success_return = '''        return explanation

    except Exception as error:'''

new_success_return = r'''        # Make XAI easier to interpret

        explanation[
            "Direction"
        ] = np.where(
            explanation[
                "Effect Toward Low"
            ] > 0,
            "Toward Low",
            "Away from Low"
        )

        absolute_values = (
            explanation[
                "Effect Toward Low"
            ]
            .abs()
        )

        if len(
            absolute_values
        ) > 0:

            high_cut = (
                absolute_values
                .quantile(
                    0.67
                )
            )

            medium_cut = (
                absolute_values
                .quantile(
                    0.33
                )
            )

            explanation[
                "Strength"
            ] = np.select(
                [
                    absolute_values
                    >= high_cut,

                    absolute_values
                    >= medium_cut
                ],
                [
                    "Strong",
                    "Moderate"
                ],
                default="Small"
            )

        else:

            explanation[
                "Strength"
            ] = "Small"

        # Make feature names easier to read

        explanation[
            "Feature"
        ] = (
            explanation[
                "Feature"
            ]
            .astype(str)
            .str.replace(
                "_",
                " ",
                regex=False
            )
        )

        # Keep numeric value for research transparency

        explanation[
            "Effect Toward Low"
        ] = (
            explanation[
                "Effect Toward Low"
            ]
            .round(4)
        )

        return explanation[
            [
                "Feature",
                "Area",
                "Direction",
                "Strength",
                "Effect Toward Low"
            ]
        ]

    except Exception as error:'''

if (
    old_success_return
    not in local_function
):

    raise RuntimeError(
        "Could not find successful XAI return block. "
        "Do not continue until this is checked."
    )

local_function = (
    local_function
    .replace(
        old_success_return,
        new_success_return,
        1
    )
)

app_text = (
    app_text[:start]
    + local_function
    + app_text[end:]
)


# Fix Gradio XAI table headers

output_start = app_text.index(
    "            explanation_output = gr.Dataframe("
)

output_end = app_text.index(
    "            recommendation_output = gr.Dataframe(",
    output_start
)

new_output_component = r'''            explanation_output = gr.Dataframe(
                headers=[
                    "Feature",
                    "Area",
                    "Direction",
                    "Strength",
                    "Effect Toward Low"
                ],
                label="XAI: Main Model Contributions",
                interactive=False
            )

'''

app_text = (
    app_text[:output_start]
    + new_output_component
    + app_text[output_end:]
)

APP_PATH.write_text(
    app_text,
    encoding="utf-8"
)

# Verify the patch really exists

check_text = APP_PATH.read_text(
    encoding="utf-8"
)

assert '"Direction"' in check_text
assert '"Strength"' in check_text

compile(
    check_text,
    str(APP_PATH),
    "exec"
)

print(
    "PASS: XAI table now includes "
    "Direction and Strength."
)

PASS: XAI table now includes Direction and Strength.


In [14]:
# 7.5 Allow missing system-generated cohort features

app_text = APP_PATH.read_text(
    encoding="utf-8"
)

old_block = '''    missing = [
        feature
        for feature in raw_features
        if feature not in df.columns
    ]

    if missing:

        raise gr.Error(
            "Missing required model columns: "
            + ", ".join(
                missing
            )
        )
'''

new_block = '''    missing = [
        feature
        for feature in raw_features
        if feature not in df.columns
    ]

    auto_filled_features = []

    for feature in missing:

        if feature in SYSTEM_GENERATED_FEATURES:

            df[
                feature
            ] = np.nan

            auto_filled_features.append(
                feature
            )

    remaining_missing = [
        feature
        for feature in raw_features
        if feature not in df.columns
    ]

    if remaining_missing:

        raise gr.Error(
            "Missing required model columns: "
            + ", ".join(
                remaining_missing
            )
        )
'''

if old_block not in app_text:

    raise RuntimeError(
        "Could not find cohort missing-column block."
    )

app_text = app_text.replace(
    old_block,
    new_block,
    1
)

APP_PATH.write_text(
    app_text,
    encoding="utf-8"
)

compile(
    app_text,
    str(APP_PATH),
    "exec"
)

print(
    "PASS: cohort upload now allows "
    "missing system-generated features."
)

PASS: cohort upload now allows missing system-generated features.


In [15]:
# 7.6 Add downloadable cohort template to the website

app_text = APP_PATH.read_text(
    encoding="utf-8"
)

# Make sure app.py knows where the template is

template_constant = '''
COHORT_TEMPLATE_PATH = (
    BASE_DIR
    / "cohort_upload_template.csv"
)
'''

marker = 'MODEL_DIR = BASE_DIR / "model"'

if "COHORT_TEMPLATE_PATH" not in app_text:

    app_text = app_text.replace(
        marker,
        marker
        + "\n"
        + template_constant,
        1
    )


# Find the Cohort Monitoring instructions

old_block = '''            gr.Markdown(
                """
Use `cohort_upload_template.csv` as the structure.

Do not upload names, IC / NRIC numbers, phone numbers, addresses, SSM numbers or social-media handles.
"""
            )

            cohort_file = gr.File(
'''

new_block = '''            gr.Markdown(
                """
### Prepare your cohort file

Download the template first, fill in the participant data, then upload the completed CSV.

**Do not include:** names, IC / NRIC numbers, phone numbers, addresses,
SSM registration numbers or social-media handles.

System-generated fields such as competency change may be left blank if unavailable.
"""
            )

            template_download = gr.DownloadButton(
                label="⬇ Download Cohort CSV Template",
                value=str(
                    COHORT_TEMPLATE_PATH
                ),
                variant="secondary"
            )

            cohort_file = gr.File(
'''

if old_block not in app_text:

    raise RuntimeError(
        "Could not find the Cohort Monitoring upload section."
    )

app_text = app_text.replace(
    old_block,
    new_block,
    1
)

APP_PATH.write_text(
    app_text,
    encoding="utf-8"
)

compile(
    app_text,
    str(APP_PATH),
    "exec"
)

print(
    "PASS: cohort template download button added."
)

PASS: cohort template download button added.


## 8. Install Deployment Libraries

In [16]:
# 8. Install deployment libraries if needed

required_imports = [
    "gradio",
    "huggingface_hub",
    "shap"
]

missing_imports = []

for package in required_imports:

    try:
        importlib.import_module(package)
    except ImportError:
        missing_imports.append(package)

if missing_imports:

    !pip -q install gradio huggingface_hub shap

print("Deployment libraries ready.")

Deployment libraries ready.


## 9. Create `requirements.txt`

In [17]:
# 9. Pin model/runtime packages

package_names = [
    "numpy",
    "pandas",
    "scikit-learn",
    "joblib",
    "shap",
    "matplotlib"
]

requirements = []

for package in package_names:

    version = importlib.metadata.version(
        package
    )

    requirements.append(
        f"{package}=={version}"
    )

pipeline = joblib.load(
    DEPLOYMENT_DIR
    / "final_early_warning_pipeline.joblib"
)

estimator = pipeline.named_steps[
    "model"
]

if (
    "xgboost"
    in estimator.__class__.__module__.lower()
):

    requirements.append(
        "xgboost=="
        + importlib.metadata.version(
            "xgboost"
        )
    )

REQUIREMENTS_PATH = (
    SPACE_DIR
    / "requirements.txt"
)

REQUIREMENTS_PATH.write_text(
    "\n".join(
        requirements
    )
    + "\n",
    encoding="utf-8"
)

print(
    REQUIREMENTS_PATH.read_text()
)

numpy==2.1.3
pandas==2.2.3
scikit-learn==1.6.1
joblib==1.5.3
shap==0.52.0
matplotlib==3.10.0
xgboost==3.4.1



## 10. Create Hugging Face `README.md`

In [18]:
# 10. Create README for the revised Space

import gradio as gr

gradio_version = gr.__version__

readme_text = f'''---
title: Ibupreneur Early Warning System
emoji: 🌸
colorFrom: pink
colorTo: pink
sdk: gradio
python_version: 3.12
sdk_version: {gradio_version}
app_file: app.py
pinned: false
---

# Ibupreneur Early-Warning System

Website-style research prototype for identifying women micro-entrepreneurs who may benefit from closer programme monitoring or support.

## Interface

- Home
- Individual Assessment
- Cohort Monitoring
- XAI & How It Works

## Quick Assessment

The Quick Assessment intentionally asks for fewer fields.

Fields that are not available are handled by the fitted preprocessing pipeline. Results based on incomplete data are labelled as preliminary screening.

## Competency Change

The competency-change z-score is not requested manually because it is a research-engineered variable derived from pre/post assessments.

## Human Oversight

The prototype must not automatically determine programme eligibility, funding, graduation or exclusion.
'''

README_PATH = (
    SPACE_DIR
    / "README.md"
)

README_PATH.write_text(
    readme_text,
    encoding="utf-8"
)

print("Created:", README_PATH)

Created: /content/drive/MyDrive/24088950_MAI_Research/outputs/huggingface_space_revised/README.md


## 11. Create Traceability Audit

In [19]:
# 11. Interface traceability

traceability_rows = []

for feature in raw_model_features:

    if feature in QUICK_FEATURES:
        ui_status = "Quick Assessment"

    elif feature in SYSTEM_GENERATED_FEATURES:
        ui_status = "System-generated / hidden"

    else:
        ui_status = "Advanced optional"

    traceability_rows.append({
        "Model Feature":
            feature,
        "Interface Status":
            ui_status,
        "Interface Label":
            display_label(feature),
        "Missing Allowed in Manual UI":
            "Yes"
            if ui_status != "Quick Assessment"
            else "No"
    })

traceability = pd.DataFrame(
    traceability_rows
)

display(
    traceability
)

TRACEABILITY_PATH = (
    SPACE_DIR
    / "interface_feature_traceability.csv"
)

traceability.to_csv(
    TRACEABILITY_PATH,
    index=False
)

,Model Feature,Interface Status,Interface Label,Missing Allowed in Manual UI
0,Age,Quick Assessment,Age,No
1,State,Advanced optional,State,Yes
2,Education Level,Advanced optional,Education Level,Yes
3,Working Status,Advanced optional,Working Status,Yes
4,Industry/Business Sector,Quick Assessment,Industry / Business Sector,No
5,Years in Business,Quick Assessment,Years in Business,No
6,Household Income,Quick Assessment,Household Income (RM),No
7,Estimated Remaining Balance,Quick Assessment,Estimated Remaining Balance (RM),No
8,ssm_registered_binary,Quick Assessment,SSM Registered,No
9,digital_platform_count,Quick Assessment,Digital Platforms Used,No


## 12. Validate Generated App

In [20]:
# 12. Syntax check

source_code = APP_PATH.read_text(
    encoding="utf-8"
)

compile(
    source_code,
    str(APP_PATH),
    "exec"
)

print("PASS: app.py syntax is valid.")

PASS: app.py syntax is valid.


## 13. Optional Local Website Preview

Set:

```python
RUN_LOCAL_PREVIEW = True
```

to see the redesigned pink website interface inside Colab.

In [ ]:
# 13. Optional local preview

RUN_LOCAL_PREVIEW = True

if RUN_LOCAL_PREVIEW:

    if str(SPACE_DIR) not in sys.path:

        sys.path.insert(
            0,
            str(SPACE_DIR)
        )

    spec = (
        importlib.util
        .spec_from_file_location(
            "ibupreneur_revised_app",
            APP_PATH
        )
    )

    app_module = (
        importlib.util
        .module_from_spec(
            spec
        )
    )

    spec.loader.exec_module(
        app_module
    )

    app_module.demo.launch(
        share=True,
        debug=True
    )

else:

    print(
        "Preview skipped. "
        "Change RUN_LOCAL_PREVIEW to True when ready."
    )

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d5f97f5daff4fd8674.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 14. Package Revised Space

In [ ]:
# 14. Create ZIP

SPACE_ZIP_BASE = (
    OUTPUT_DIR
    / "ibupreneur_huggingface_space_revised"
)

SPACE_ZIP_PATH = shutil.make_archive(
    str(SPACE_ZIP_BASE),
    "zip",
    root_dir=SPACE_DIR
)

print("ZIP:", SPACE_ZIP_PATH)

# Deploy to Existing Hugging Face Space

## 15. Clone Existing Repository

Your existing Space:

```text
Izzzahfauzan/EarlyWarningSystem
```

Direct push is disabled until authentication is configured.

In [ ]:
# 15. Clone existing Hugging Face Space

HF_REPO_URL = (
    "https://huggingface.co/spaces/"
    "Izzzahfauzan/EarlyWarningSystem"
)

HF_REPO_DIR = Path(
    "/content/EarlyWarningSystem"
)

CLONE_EXISTING_REPO = True

if CLONE_EXISTING_REPO:

    if HF_REPO_DIR.exists():

        shutil.rmtree(
            HF_REPO_DIR
        )

    %cd /content
    !git clone {HF_REPO_URL}

    print(
        "Repository cloned to:",
        HF_REPO_DIR
    )

else:

    print(
        "Clone skipped. "
        "Set CLONE_EXISTING_REPO = True when ready."
    )

## 16. Copy Revised Website into Repository

In [ ]:
# 16. Copy generated Space into cloned repository

COPY_TO_REPO = False

if COPY_TO_REPO:

    if not HF_REPO_DIR.exists():

        raise FileNotFoundError(
            "Clone the Hugging Face Space first."
        )

    for item in SPACE_DIR.iterdir():

        destination = (
            HF_REPO_DIR
            / item.name
        )

        if item.is_dir():

            if destination.exists():

                shutil.rmtree(
                    destination
                )

            shutil.copytree(
                item,
                destination
            )

        else:

            shutil.copy2(
                item,
                destination
            )

    print(
        "Revised website copied to repository."
    )

else:

    print(
        "Copy skipped."
    )

## 17. Authentication and Push

Keep the token in **Colab Secrets** as `HF_TOKEN`.

Do not paste it directly into notebook code.

In [ ]:
# 17. Authenticate safely

AUTHENTICATE_HF = True

if AUTHENTICATE_HF:

    from google.colab import userdata
    from huggingface_hub import login

    HF_TOKEN = userdata.get(
        "HF_TOKEN"
    )

    login(
        token=HF_TOKEN,
        add_to_git_credential=False
    )

    print(
        "Hugging Face authentication successful."
    )

else:

    print(
        "Authentication skipped."
    )

In [ ]:
# 17.1 Commit and push

PUSH_TO_HF = True

if PUSH_TO_HF:

    if not HF_REPO_DIR.exists():

        raise FileNotFoundError(
            "Hugging Face repository is not cloned."
        )

    %cd /content/EarlyWarningSystem

    !git config user.name "Izzzahfauzan"
    !git add .
    !git commit -m "add ZeroGPU compatibility"
    !git push

else:

    print(
        "Push skipped."
    )

In [ ]:
from pathlib import Path
import shutil

SPACE_DIR = Path(
    "/content/drive/MyDrive/24088950_MAI_Research/"
    "outputs/huggingface_space_revised"
)

HF_REPO_DIR = Path(
    "/content/EarlyWarningSystem"
)

print("Source exists:", SPACE_DIR.exists())
print("Repo exists:", HF_REPO_DIR.exists())

In [ ]:
for item in SPACE_DIR.iterdir():

    destination = (
        HF_REPO_DIR
        / item.name
    )

    if item.is_dir():

        if destination.exists():
            shutil.rmtree(
                destination
            )

        shutil.copytree(
            item,
            destination
        )

    else:

        shutil.copy2(
            item,
            destination
        )

print(
    "Revised files copied into Hugging Face repo."
)

In [ ]:
%cd /content/EarlyWarningSystem

!git status

In [ ]:
%cd /content/EarlyWarningSystem

!git config user.name "Izzzahfauzan"
!git config user.email "24088950@siswa.um.edu.my"

In [ ]:
!git add .

!git status

In [ ]:
!git commit -m "update early warning website prototype"

In [ ]:
!git push origin main

## 18. Final Checklist

Before deployment, confirm:

- Quick Assessment feels short enough;
- competency z-score is not requested manually;
- optional Advanced Information is collapsed;
- pink theme is visible;
- the UI feels like a website;
- High / Moderate / Low appear as visual trajectory cards;
- probability values are shown;
- result meaning is written in plain language;
- XAI explains the main model contributions;
- suggested support is based on actionable programme areas;
- incomplete manual assessments are labelled preliminary;
- cohort upload still uses the full prepared model dataset;
- human oversight disclaimer is visible.

In [ ]:
# 18. Final summary

print(
    "REVISED NOTEBOOK 07 COMPLETE"
)

print()

print(
    "Quick fields:",
    len(QUICK_FEATURES)
)

print(
    "Advanced optional fields:",
    len(ADVANCED_FEATURES)
)

print(
    "System-generated hidden fields:",
    len(SYSTEM_GENERATED_FEATURES)
)

print()

print(
    "Revised Space folder:",
    SPACE_DIR
)

print()

print(
    "Revised Space ZIP:",
    SPACE_ZIP_PATH
)